In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
!pip install LTNtorch

In [10]:
!pip uninstall ltn

In [ ]:
import sys

!{sys.executable} -m pip install numpy pandas matplotlib
!{sys.executable} -m pip install torch torchvision
!{sys.executable} -m pip install torchmetrics
!{sys.executable} -m pip install LTNtorch

: 

In [2]:

!pip install numpy pandas matplotlib
!pip install torch
!pip install torchvision
!pip install torchmetrics
!pip install LTNtorch

testes

In [ ]:


import ltn
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from PIL import Image
import os

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
#from tqdm import tqdm

# !pip install torchvision
import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
from datetime import datetime
import importlib
#from utils.axioms import compute_axioms, get_number_of_final_classes, get_CLASSES
#from utils.occlusion import occlusion_map, show_occlusion
#print(f"final classes {get_number_of_final_classes()}")

# !pip install torchmetrics
import torchmetrics

""" HYPERPARAMETERS """


DATA_ROOT = Path("/home/sofia/Desktop/datasetVCB")
BATCH_SIZE = 32
COMPLEXITY_DATASET = 1
EPOCHS = 10
TOTAL_EXAMPLES = 15000
TRAIN_TEST_LIMIT = 12000
LEARNING_RATE = 0.001
csv_path = "results.csv"
txt_path = "stats.txt"
confusion_matrix_path = "conf_matrix.txt"

final classes 1


In [ ]:
BATCH_SIZE = 32
LIMIT = 1500
EPOCHS = 150
EARLY_STOPPING_DELTA = 0.005
PATIENCE = 10
MODEL_SAVING_INTERVAL = 20

In [5]:
import torch

print("torch works")
x = torch.randn(1000,1000)
print(x.mean())

torch works
tensor(0.0005)


CNN with multiple Loss

In [ ]:
#from utils.axioms import compute_axioms, get_number_of_final_classes, get_CLASSES
import utils.axioms as axioms
import utils.occlusion as occlusion
importlib.reload(axioms)
importlib.reload(occlusion)

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = axioms.get_CLASSES()
final_classes = axioms.get_number_of_final_classes()

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""

class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
       #self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

        #3 conv 43624, 4 conv 18432
       self.dense1 = nn.Linear(43264, 128)
       self.dense2 = nn.Linear(128, 64)

       self.concept_layer = nn.Linear(64, 32)
       self.dense3 = nn.Linear(32, 16)
       self.output_layer = nn.Linear(16, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = F.relu(self.dense3(y))

       x = self.output_layer(x)

       result = torch.cat([x, y], dim=1)
       return result

      
'''
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes, name="cnn_model"):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.conv3 = nn.Conv2d(32, 64, 3)

        self.pool = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

        self.concept_layer = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        y = F.relu(self.concept_layer(x))
        x = self.output_layer(y)

        return torch.cat([x, y], dim=1)

'''
class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        #print(selected.shape)
        return selected  # [batch]




'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)


# Predicates
#final classes = 4
logits_model = CNN(in_channels=3, num_classes=final_classes)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)
#missCounter = 0
classification_loss_fn = nn.BCEWithLogitsLoss()

p = ltn.Predicate(PredicateModel())


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class Counter:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.value = 0.0

    def update(self, value):
        self.value = value

    def result(self):
        return self.value

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0


    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()
    

class RunningAverage:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.total = 0
        self.count = 0

    def update(self, value):
        self.total += value
        self.count += 1

    
    def result(self):
        if self.count == 0:
            return 0
        return self.total / self.count


metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
    "train_loss": RunningAverage(),
    "test_loss": RunningAverage(),
    "alpha": Counter()
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

#optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list, optimizer, alpha=0.5):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    # first 4 logits correspond to classes
    class_logits = logits[:, :final_classes]

    # only first 4 labels
    class_labels = labels_list[:final_classes]
    labels_tensor = torch.stack(class_labels, dim=1).float()

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat = axioms.compute_axioms(logits, *labels_list, p=p)

    ltn_loss = 1.0 - sat

    # ----- Combined loss -----
    loss = alpha * classification_loss + (1 - alpha) * ltn_loss

    loss.backward()
    optimizer.step()

    print(f"Sat Function Result: {sat.item()}, Class_loss: {classification_loss}, ltn-loss: {ltn_loss}, loss:{loss} ")

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())
    metrics_dict["train_loss"].update(classification_loss.item())

    #predictions = logits

    calculate_accuracy_for_each_class(logits, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list, alpha=0.5):
    logits_model.eval()

    logits = logits_model(features_param)
        
    # first 4 logits correspond to classes
    class_logits = logits[:, :final_classes]

    # only first 4 labels
    class_labels = labels_list[:final_classes]
    labels_tensor = torch.stack(class_labels, dim=1).float()

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat_kb = axioms.compute_axioms(logits, *labels_list, p=p)

    ltn_loss = 1.0 - sat_kb

    # ----- Combined loss -----
    loss = sat_kb * classification_loss + (1 - alpha) * ltn_loss


    metrics_dict["test_sat_kb"].update(sat_kb.item())
    metrics_dict["test_loss"].update(loss.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file, model, epochs_completed, warmup_epochs, final_epochs, final_alpha, patience):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")

    txt_file.write(f"Number of epochs completed: {epochs_completed}\n")
    txt_file.write(f"Warmup epochs: {warmup_epochs}; final_epochs: {final_epochs}; final alpha: {final_alpha}; patience: {patience}")
    txt_file.write("\n")

    if model is not None:
        txt_file.write("\nModel Architecture:\n")
        txt_file.write(str(model))
        txt_file.write("\n\n")


#final_epochs: epochs with final alpha
def new_calculate_alpha(epoch= 0, warmup_epochs = 10, final_alpha = 0.25, final_epochs=30):
    alpha = 1.0

    if epoch < warmup_epochs:
        alpha = 1.0
    elif epoch < EPOCHS - final_epochs:
        progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs-final_epochs)
        alpha = 1.0 - progress * (1.0 - final_alpha)
    else:
        alpha = final_alpha

    return alpha


""" SCRIPT """
LIMIT = 1500
EPOCHS = 120 #isto serve para definir quao gradualmente o alpha é modeficado, 150 era fixe para o conhecimento original

#EARLY_STOPPING_DELTA = 0.005
#PATIENCE = 20
MODEL_SAVING_INTERVAL = 10
LEARNING_RATE=0.001
output_dir = ""

def train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, patience=20, early_stopping_delta=0.001, learning_rate=0.001, train_model_number=0):

    #folder
    run_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_dir = Path(f"outputs{train_model_number}_{run_name}")
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / "metrics.csv"
    txt_path = output_dir / "stats.txt"
    txt_path_notes = output_dir / "notas.txt"

    metrics_dict.update(class_train_accuracy_metric)

    optimizer = torch.optim.Adam(logits_model.parameters(), lr=learning_rate)

    best_sat = 0.0
    epochs_no_improve = 0

    epochs_completed = 0

    # scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
    template = "Epoch {}"
    for metrics_label in metrics_dict.keys():
        template += ", %s: {:.5f}" % metrics_label
    if csv_path is not None:

        csv_file = open(csv_path, "w+")
        headers = ",".join(["Epoch", "Alpha"] + list(metrics_dict.keys())) #sem o alpha
        csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 2)]) #era +1
        csv_file.write(headers + "\n")

    for epoch in range(LIMIT):
        print(f"starting epoch {epoch}")
        #reset metrics
        for metrics in metrics_dict.values():
            metrics.reset_state()

        alpha = new_calculate_alpha(epoch, warmup_epochs, final_alpha, final_epochs)
        #alpha = 0.5
        metrics_dict["alpha"].update(alpha)

        print(f"alpha: {alpha}")

        for batch_elements in train_loader:
            features_param, labels_list = __get_elements_from_batch(batch_elements)

            #print("labels list")
            #print(labels_list.shape)

            # Move to device if needed
            features_param = features_param.to(device)
            labels_list = [lbl.to(device) for lbl in labels_list]

            #print("train step")
            train_step(features_param, *labels_list, optimizer=optimizer, alpha=alpha)


        for batch_elements_test in test_loader:
            features_param, labels_list = __get_elements_from_batch(batch_elements_test)

            # Move to device if needed
            features_param = features_param.to(device)
            labels_list = [lbl.to(device) for lbl in labels_list]

            test_step(features_param, *labels_list, alpha=alpha)

        metrics_results = [metrics.result() for metrics in metrics_dict.values()]
        print(template.format(epoch, *metrics_results))
        if csv_path is not None:
            #csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
            csv_file.write(csv_template.format(epoch, alpha, *metrics_results) + "\n")
            csv_file.flush()

        #early stopping condition
        current_sat = metrics_dict["test_sat_kb"].result()
        new_sat = best_sat + early_stopping_delta

        print(f"best_sat: {best_sat}, current_sat: {current_sat}, number of no improve runs:{epochs_no_improve}/{patience}")

        if current_sat > new_sat:
            best_sat = current_sat
            epochs_no_improve = 0
            print("\n\n\n CHANGED BEST SAT \n\n\n")
        elif epoch > 0:
            #print("\n\n\n CHANGED BEST SAT \n\n\n")
            epochs_no_improve += 1


        if epochs_no_improve >= patience:
            print("EARLY STOPPING TRIGGERED")
            epochs_completed = epoch
            break

        #model saving
        if epoch != 0 and epoch % MODEL_SAVING_INTERVAL == 0:
            print(f"SAVING MODEL AT EPOCH {epoch}")
            torch.save(logits_model.state_dict(), output_dir / f"model_epoch_{epoch}.pth")
        


    print("training finished, starting to save the model")
    txt_file = open(txt_path, "w+")
    notas = open(txt_path_notes, "w+")
    write_results_description_in_txt(txt_file, logits_model, epochs_completed,warmup_epochs, final_epochs, final_alpha, patience)

    torch.save(logits_model.state_dict(), output_dir / f"model_epoch_{epoch}.pth")

    if csv_path is not None:
        csv_file.close()
        txt_file.close()
        notas.close()
    

    


#train loop

train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, patience=20, early_stopping_delta=0.005, learning_rate=LEARNING_RATE, train_model_number=0)

starting epoch 0
alpha: 1.0
Sat Function Result: 0.5126914381980896, Class_loss: 0.6779954433441162, ltn-loss: 0.4873085618019104, loss:0.6779954433441162 
Sat Function Result: 0.5103347301483154, Class_loss: 0.6806384325027466, ltn-loss: 0.48966526985168457, loss:0.6806384325027466 
Sat Function Result: 0.5058821439743042, Class_loss: 0.6715258359909058, ltn-loss: 0.4941178560256958, loss:0.6715258359909058 
Sat Function Result: 0.495052695274353, Class_loss: 0.6290139555931091, ltn-loss: 0.504947304725647, loss:0.6290139555931091 
Sat Function Result: 0.4702356457710266, Class_loss: 0.6234406232833862, ltn-loss: 0.5297643542289734, loss:0.6234406232833862 
Sat Function Result: 0.42184919118881226, Class_loss: 0.6007955074310303, ltn-loss: 0.5781508088111877, loss:0.6007955074310303 
Sat Function Result: 0.38530993461608887, Class_loss: 0.6423022150993347, ltn-loss: 0.6146900653839111, loss:0.6423022150993347 
Sat Function Result: 0.3509523272514343, Class_loss: 0.6047896146774292, lt

In [30]:
import utils.axioms as axioms
import utils.occlusion as occlusion
importlib.reload(axioms)
importlib.reload(occlusion)


output_dir = "outputs0_2026-03-21_13-39-46"
model_path = "outputs0_2026-03-21_13-39-46/model_epoch_31.pth"


logits_model.load_state_dict(torch.load(model_path))

print(test_dataset[0])

img, _ = test_dataset[0]

heatmap = occlusion.occlusion_map(logits_model, img, class_idx=0)
occlusion.show_occlusion(img, heatmap, output_dir)

(tensor([[[0.3961, 0.3961, 0.3882,  ..., 0.4471, 0.4471, 0.4471],
         [0.3961, 0.3961, 0.3882,  ..., 0.4510, 0.4510, 0.4510],
         [0.3961, 0.3961, 0.3961,  ..., 0.4510, 0.4510, 0.4510],
         ...,
         [0.2980, 0.2863, 0.2745,  ..., 0.2667, 0.2471, 0.2392],
         [0.2824, 0.2784, 0.2784,  ..., 0.2745, 0.2667, 0.2627],
         [0.2706, 0.2745, 0.2824,  ..., 0.2549, 0.2588, 0.2549]],

        [[0.4980, 0.4980, 0.5020,  ..., 0.5020, 0.5020, 0.5020],
         [0.4980, 0.4980, 0.5020,  ..., 0.5059, 0.5059, 0.5059],
         [0.4980, 0.4980, 0.4980,  ..., 0.5059, 0.5059, 0.5059],
         ...,
         [0.2706, 0.2588, 0.2471,  ..., 0.2667, 0.2549, 0.2471],
         [0.2549, 0.2510, 0.2510,  ..., 0.2745, 0.2745, 0.2784],
         [0.2431, 0.2471, 0.2549,  ..., 0.2627, 0.2667, 0.2706]],

        [[0.4941, 0.4941, 0.4941,  ..., 0.5020, 0.5020, 0.5020],
         [0.4941, 0.4941, 0.4941,  ..., 0.5059, 0.5059, 0.5059],
         [0.4941, 0.4941, 0.4941,  ..., 0.5059, 0.5059, 0

old code

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]
final_classes = 4

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""

class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
       #self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

        #3 conv 43624, 4 conv 18432
       self.dense1 = nn.Linear(43264, 128)#com 3 conv, 43264
       self.dense2 = nn.Linear(128, 64)

       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result

      
'''
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes, name="cnn_model"):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.conv3 = nn.Conv2d(32, 64, 3)

        self.pool = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

        self.concept_layer = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        y = F.relu(self.concept_layer(x))
        x = self.output_layer(y)

        return torch.cat([x, y], dim=1)

'''
class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        #print(selected.shape)
        return selected  # [batch]




'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)


# Predicates
#final classes = 4
logits_model = CNN(in_channels=3, num_classes=final_classes)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)
#missCounter = 0
classification_loss_fn = nn.BCEWithLogitsLoss()

p = ltn.Predicate(PredicateModel())


# Constants
#final classes
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)

#concepts
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)



#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)

#print("before axioms")

#missCounter = {"count": 0}

#axioms
def axioms(logits, *args):

    #logits = logits_model(features_param)

    x = ltn.Variable("x", logits)
    x_hotel = ltn.Variable("x_hotel", logits[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", logits[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", logits[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", logits[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", logits[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", logits[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", logits[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        logits[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    def safe_forall(var, formula, default_value=0.5):
    # Check if variable has zero elements
        
        if var.value.shape[0] == 0:  # no elements in the batch
            # Return a neutral truth value (0.5)
            print("\n\n\n\n\n\n\n\n miss \n\n\n\n\n\n")

            #missCounter += 1
            return ltn.Constant(torch.tensor(default_value, device=logits.device))
        else:
            return Forall(var, formula)



    axioms = [
        # Grounded Predicates

        safe_forall(x_hotel, p(x_hotel, class_hotel)),
        safe_forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        safe_forall(x_cafe, p(x_cafe, class_cafe)),
        safe_forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        safe_forall(x_store, p(x_store, class_store)),
        safe_forall(x_not_store, Not(p(x_not_store, class_store))),
        safe_forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        safe_forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        safe_forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        safe_forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        safe_forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        safe_forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    #print("values of axioms")
    #for a in axioms:
      #print(a.value)

    sat_level = formula_aggregator(*axioms)

    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class Counter:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.value = 0.0

    def update(self, value):
        self.value = value

    def result(self):
        return self.value

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0


    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()
    

class RunningAverage:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.total = 0
        self.count = 0

    def update(self, value):
        self.total += value
        self.count += 1

    
    def result(self):
        if self.count == 0:
            return 0
        return self.total / self.count


metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
    "train_loss": RunningAverage(),
    "test_loss": RunningAverage(),
    "alpha": Counter()
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

#optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list, optimizer, alpha=0.5):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    # first 4 logits correspond to classes
    class_logits = logits[:, :4]

    # only first 4 labels
    class_labels = labels_list[:4]
    labels_tensor = torch.stack(class_labels, dim=1).float()

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat = axioms(logits, *labels_list)

    ltn_loss = 1.0 - sat

    # ----- Combined loss -----
    loss = alpha * classification_loss + (1 - alpha) * ltn_loss

    loss.backward()
    optimizer.step()

    print(f"Sat Function Result: {sat.item()}, Class_loss: {classification_loss}, ltn-loss: {ltn_loss}, loss:{loss} ")

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())
    metrics_dict["train_loss"].update(classification_loss.item())

    #predictions = logits

    calculate_accuracy_for_each_class(logits, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list, alpha=0.5):
    logits_model.eval()

    logits = logits_model(features_param)
        
    # first 4 logits correspond to classes
    class_logits = logits[:, :4]

    # only first 4 labels
    class_labels = labels_list[:4]
    labels_tensor = torch.stack(class_labels, dim=1).float()

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat_kb = axioms(logits, *labels_list)

    ltn_loss = 1.0 - sat_kb

    # ----- Combined loss -----
    loss = sat_kb * classification_loss + (1 - alpha) * ltn_loss


    metrics_dict["test_sat_kb"].update(sat_kb.item())
    metrics_dict["test_loss"].update(loss.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file, model):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")

    if model is not None:
        txt_file.write("\nModel Architecture:\n")
        txt_file.write(str(model))
        txt_file.write("\n\n")


#final_epochs: epochs with final alpha
def new_calculate_alpha(epoch= 0, warmup_epochs = 10, final_alpha = 0.25, final_epochs=30):
    alpha = 1.0

    if epoch < warmup_epochs:
        alpha = 1.0
    elif epoch < EPOCHS - final_epochs:
        progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs-final_epochs)
        alpha = 1.0 - progress * (1.0 - final_alpha)
    else:
        alpha = final_alpha

    return alpha


""" SCRIPT """
LIMIT = 1500
EPOCHS = 80 #era 150
#EARLY_STOPPING_DELTA = 0.005
#PATIENCE = 20
MODEL_SAVING_INTERVAL = 10

def train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, patience=20, early_stopping_delta=0.001, learning_rate=0.001, train_model_number=0):

    #folder
    run_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    OUTPUT_DIR = Path(f"outputs{train_model_number}_{run_name}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    csv_path = OUTPUT_DIR / "metrics.csv"
    txt_path = OUTPUT_DIR / "stats.txt"

    metrics_dict.update(class_train_accuracy_metric)

    optimizer = torch.optim.Adam(logits_model.parameters(), lr=learning_rate)

    best_sat = 0.0
    epochs_no_improve = 0

    # scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
    template = "Epoch {}"
    for metrics_label in metrics_dict.keys():
        template += ", %s: {:.5f}" % metrics_label
    if csv_path is not None:
        txt_file = open(txt_path, "w+")
        write_results_description_in_txt(txt_file, logits_model)

        csv_file = open(csv_path, "w+")
        headers = ",".join(["Epoch", "Alpha"] + list(metrics_dict.keys())) #sem o alpha
        csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 2)]) #era +1
        csv_file.write(headers + "\n")

    for epoch in range(LIMIT):
        print(f"starting epoch {epoch}")
        #reset metrics
        for metrics in metrics_dict.values():
            metrics.reset_state()

        #alpha = new_calculate_alpha(epoch, warmup_epochs, final_alpha, final_epochs)
        alpha = 0.5
        metrics_dict["alpha"].update(alpha)

        print(f"alpha: {alpha}")

        for batch_elements in train_loader:
            features_param, labels_list = __get_elements_from_batch(batch_elements)

            #print("labels list")
            #print(labels_list.shape)

            # Move to device if needed
            features_param = features_param.to(device)
            labels_list = [lbl.to(device) for lbl in labels_list]

            #print("train step")
            train_step(features_param, *labels_list, optimizer=optimizer, alpha=alpha)


        for batch_elements_test in test_loader:
            features_param, labels_list = __get_elements_from_batch(batch_elements_test)

            # Move to device if needed
            features_param = features_param.to(device)
            labels_list = [lbl.to(device) for lbl in labels_list]

            test_step(features_param, *labels_list, alpha=alpha)

        metrics_results = [metrics.result() for metrics in metrics_dict.values()]
        print(template.format(epoch, *metrics_results))
        if csv_path is not None:
            #csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
            csv_file.write(csv_template.format(epoch, alpha, *metrics_results) + "\n")
            csv_file.flush()

        #early stopping condition
        current_sat = metrics_dict["test_sat_kb"].result()
        new_sat = best_sat + early_stopping_delta

        print(f"best_sat: {best_sat}, current_sat: {current_sat}")

        if current_sat > new_sat:
            best_sat = current_sat
            epochs_no_improve = 0
            print("\n\n\n CHANGED BEST SAT \n\n\n")
        elif epoch > 0:
            #print("\n\n\n CHANGED BEST SAT \n\n\n")
            epochs_no_improve += 1


        if epochs_no_improve >= patience:
            print("EARLY STOPPING TRIGGERED")
            break

        #model saving
        if epoch != 0 and epoch % MODEL_SAVING_INTERVAL == 0:
            print(f"SAVING MODEL AT EPOCH {epoch}")
            torch.save(logits_model.state_dict(), OUTPUT_DIR / f"model_epoch_{epoch}.pth")
        




    print("training finished, starting to save the model")
    torch.save(logits_model.state_dict(), OUTPUT_DIR / f"model_epoch_{epoch}.pth")

    if csv_path is not None:
        csv_file.close()
        txt_file.close()


#train loop

train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, patience=10, early_stopping_delta=0.005, learning_rate=LEARNING_RATE, train_model_number=0)

script completo

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""

class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
       #self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

        #3 conv 43624, 4 conv 18432
       self.dense1 = nn.Linear(43264, 128)#com 3 conv, 43264
       self.dense2 = nn.Linear(128, 64)

       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result

      
'''
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes, name="cnn_model"):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.conv3 = nn.Conv2d(32, 64, 3)

        self.pool = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

        self.concept_layer = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        y = F.relu(self.concept_layer(x))
        x = self.output_layer(y)

        return torch.cat([x, y], dim=1)

'''
class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        #print(selected.shape)
        return selected  # [batch]




'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)


# Predicates
logits_model = CNN(in_channels=3, num_classes=4)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)
#missCounter = 0

p = ltn.Predicate(PredicateModel())


# Constants
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)



#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)

#print("before axioms")

#missCounter = {"count": 0}

#axioms
def axioms(logits, *args):

    #logits = logits_model(features_param)

    x = ltn.Variable("x", logits)
    x_hotel = ltn.Variable("x_hotel", logits[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", logits[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", logits[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", logits[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", logits[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", logits[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", logits[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        logits[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    def safe_forall(var, formula, default_value=0.5):
    # Check if variable has zero elements
        
        if var.value.shape[0] == 0:  # no elements in the batch
            # Return a neutral truth value (0.5)
            print("\n\n\n\n\n\n\n\n miss \n\n\n\n\n\n")

            #missCounter += 1
            return ltn.Constant(torch.tensor(default_value, device=logits.device))
        else:
            return Forall(var, formula)



    axioms = [
        # Grounded Predicates

        safe_forall(x_hotel, p(x_hotel, class_hotel)),
        safe_forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        safe_forall(x_cafe, p(x_cafe, class_cafe)),
        safe_forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        safe_forall(x_store, p(x_store, class_store)),
        safe_forall(x_not_store, Not(p(x_not_store, class_store))),
        safe_forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        safe_forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        safe_forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        safe_forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        safe_forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        safe_forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    #print("values of axioms")
    #for a in axioms:
      #print(a.value)

    sat_level = formula_aggregator(*axioms)

    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    #print("axioms")
    # Forward through axioms (LTN)
    sat = axioms(logits, *labels_list)

    #print("sat")
    #print(sat)
    loss = 1.0 - sat
    #print("loss")
    #print(loss)
    #print(sat.value)

    # Backprop
    #print ("before backward")
    loss.backward()
    #print("after backward")
    optimizer.step()
    #print("after optimizer")

    print("Sat Function Result:", sat.item())

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())

    # Predictions
    #predictions = logits_model(features_param)
    predictions = logits

    calculate_accuracy_for_each_class(predictions, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list):
    logits_model.eval()

    logits = logits_model(features_param)

    sat_kb = axioms(logits, *labels_list)

    metrics_dict["test_sat_kb"].update(sat_kb.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file, model):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")

    if model is not None:
        txt_file.write("\nModel Architecture:\n")
        txt_file.write(str(model))
        txt_file.write("\n\n")




""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file, logits_model)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()

    print("after metrics")

    for batch_elements in train_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        #print("train step")
        train_step(features_param, *labels_list)


    for batch_elements_test in test_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements_test)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

print("training finished, starting to save the model")
torch.save(logits_model.state_dict(), f"model_epoch_{epoch}.pth")

if csv_path is not None:
    csv_file.close()
    txt_file.close()

In [21]:
torch.save(logits_model.state_dict(), f"model_epoch_{epoch}.pth")

com resnet18

In [18]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""
class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

       self.dense1 = nn.Linear(43264, 128)
       self.dense2 = nn.Linear(128, 64)
       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result


class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        #print(selected.shape)
        return selected  # [batch]
    

class ResNetLTN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = models.resnet18(weights="IMAGENET1K_V1")

        # remove default classifier
        self.backbone.fc = nn.Identity()

        self.concept_layer = nn.Linear(512, 64)
        self.output_layer = nn.Linear(64, num_classes)

    def forward(self, x):

        features = self.backbone(x)   # 512 feature vector

        concepts = torch.relu(self.concept_layer(features))
        logits = self.output_layer(concepts)

        result = torch.cat([logits, concepts], dim=1)

        return result




'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)
resnet = models.resnet18(weights="IMAGENET1K_V1")

# Predicates
logits_model = ResNetLTN(num_classes=4)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)
#missCounter = 0

p = ltn.Predicate(PredicateModel())


# Constants
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)



#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)

#print("before axioms")

#missCounter = {"count": 0}

#axioms
def axioms(logits, *args):

    #logits = logits_model(features_param)

    x = ltn.Variable("x", logits)
    x_hotel = ltn.Variable("x_hotel", logits[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", logits[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", logits[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", logits[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", logits[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", logits[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", logits[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        logits[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    def safe_forall(var, formula, default_value=0.5):
    # Check if variable has zero elements
        
        if var.value.shape[0] == 0:  # no elements in the batch
            # Return a neutral truth value (0.5)
            print("\n\n\n\n\n\n\n\n miss \n\n\n\n\n\n")

            #missCounter += 1
            return torch.tensor(default_value, device="cuda" if torch.cuda.is_available() else "cpu", requires_grad=True)
        else:
            return Forall(var, formula)



    axioms = [
        # Grounded Predicates

        safe_forall(x_hotel, p(x_hotel, class_hotel)),
        safe_forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        safe_forall(x_cafe, p(x_cafe, class_cafe)),
        safe_forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        safe_forall(x_store, p(x_store, class_store)),
        safe_forall(x_not_store, Not(p(x_not_store, class_store))),
        safe_forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        safe_forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        safe_forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        safe_forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        safe_forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        safe_forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    #print("values of axioms")
    #for a in axioms:
      #print(a.value)

    sat_level = formula_aggregator(*axioms)

    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    # Forward through axioms (LTN)
    sat = axioms(logits, *labels_list)

    loss = 1.0 - sat

    # Backprop
    loss.backward()
    optimizer.step()

    print("Sat Function Result:", sat.item())

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())

    # Predictions
    predictions = logits_model(features_param)

    calculate_accuracy_for_each_class(predictions, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list):
    logits_model.eval()

    logits = logits_model(features_param)

    sat_kb = axioms(logits, *labels_list)

    metrics_dict["test_sat_kb"].update(sat_kb.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")




""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()

    for batch_elements in train_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        train_step(features_param, *labels_list)


    for batch_elements_test in test_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements_test)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()


torch.save(logits_model.state_dict(), f"model_epoch_{epoch}.pth")

if csv_path is not None:
    csv_file.close()
    txt_file.close()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/sofia/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

baseline

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""
class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

       self.dense1 = nn.Linear(43264, 128)
       self.dense2 = nn.Linear(128, 64)
       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       x = F.relu(self.concept_layer(x))
       x = self.output_layer(x)

       #result = torch.cat([x, y], dim=1)
       return x





'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)

criterion = torch.nn.BCEWithLogitsLoss()

# Predicates
logits_model = CNN(in_channels=3, num_classes=4)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)


#print("before axioms")




'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class RunningAverage:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.total = 0
        self.count = 0

    def update(self, value):
        self.total += value
        self.count += 1

    
    def result(self):
        if self.count == 0:
            return 0
        return self.total / self.count

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_loss": RunningAverage(),
    "test_loss": RunningAverage()
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, labels):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    loss = criterion(logits, labels)

    print(f"current loss: {loss}")

    loss.backward()
    optimizer.step()

    metrics_dict["train_loss"].update(loss.item())

    calculate_accuracy_for_each_class(logits, labels)


@torch.no_grad()
def test_step(features, labels):

    logits_model.eval()

    logits = logits_model(features)

    loss = criterion(logits, labels)

    metrics_dict["test_loss"].update(loss.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):

    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):

        class_labels = labels_param[:, i]

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")




""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()


    for features_param, labels in train_loader:

      features_param = features_param.to(device)
      labels = labels.to(device)

      train_step(features_param, labels)


    for features_param, labels in test_loader:

        features_param = features_param.to(device)
        labels = labels.to(device)

        test_step(features_param, labels)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

torch.save(logits_model.state_dict(), f"model_baseline_epoch_{epoch}.pth")

if csv_path is not None:
    csv_file.close()
    txt_file.close()

starting epoch 0
current loss: 0.6771466732025146
current loss: 0.6567100286483765
current loss: 0.5981974601745605
current loss: 0.5395768880844116
current loss: 0.5126150846481323
current loss: 0.66026771068573
current loss: 0.616314709186554
current loss: 0.5390615463256836
current loss: 0.6431362628936768
current loss: 0.4919160008430481
current loss: 0.5806919932365417
current loss: 0.5475058555603027
current loss: 0.6184510588645935
current loss: 0.5247862339019775
current loss: 0.5619179010391235
current loss: 0.5090570449829102
current loss: 0.47366923093795776
current loss: 0.4436870813369751
current loss: 0.6050456166267395
current loss: 0.5044364333152771
current loss: 0.5341821312904358
current loss: 0.49942341446876526
current loss: 0.54839026927948
current loss: 0.4972171187400818
current loss: 0.5163198709487915


começo do codigo

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)



CNN model

In [ ]:
#print(train_df["Cafe"])

"""CNN MODEL"""
class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

       self.dense1 = nn.Linear(43264, 128)
       self.dense2 = nn.Linear(128, 64)
       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result


class PredicateModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.activation = nn.Sigmoid()

    def forward(self, x, class_idx):
        logits = self.model(x)              # [batch, num_classes]
        probs = self.activation(logits)     # [0,1]
        print("shape in prpbs")
        print(probs.shape)
        #print("shape in predicate model")
        #print(probs[:, class_idx].squeeze().shape)

        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(x.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]






LTN Model

In [ ]:


# Predicates
logits_model = CNN(in_channels=3, num_classes=4)

# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# move model to device
logits_model = logits_model.to(device)

p = ltn.Predicate(PredicateModel(logits_model))

#print("before constants")

# Constants
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)


#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)


#axioms
def axioms(features_param, *args):

    x = ltn.Variable("x", features_param)
    x_hotel = ltn.Variable("x_hotel", features_param[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", features_param[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", features_param[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", features_param[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    axioms = [
        # Grounded Predicates


        Forall(x_hotel, p(x_hotel, class_hotel)),
        Forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        Forall(x_cafe, p(x_cafe, class_cafe)),
        Forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        Forall(x_store, p(x_store, class_store)),
        Forall(x_not_store, Not(p(x_not_store, class_store))),
        Forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        Forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        Forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        Forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    sat_level = formula_aggregator(*axioms)
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

print("before for loop")

for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break



before for loop
shape in prpbs
torch.Size([7, 36])
shape in prpbs
torch.Size([25, 36])
shape in prpbs
torch.Size([7, 36])
shape in prpbs
torch.Size([25, 36])
shape in prpbs
torch.Size([3, 36])
shape in prpbs
torch.Size([29, 36])
shape in prpbs
torch.Size([4, 36])
shape in prpbs
torch.Size([28, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
Initial sat level 0.51488


In [ ]:
""" TRAINING FUNCTIONS """

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list):
    logits_model.train()

    optimizer.zero_grad()

    # Forward through axioms (LTN)
    sat = axioms(features_param, *labels_list)

    loss = 1.0 - sat

    # Backprop
    loss.backward()
    optimizer.step()

    print("Sat Function Result:", sat.item())

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())

    # Predictions
    predictions = logits_model(features_param)

    calculate_accuracy_for_each_class(predictions, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list):
    logits_model.eval()

    sat_kb = axioms(features_param, *labels_list)

    metrics_dict["test_sat_kb"].update(sat_kb.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")

In [ ]:
""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()

    for batch_elements in train_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        train_step(features_param, *labels_list)


    for batch_elements_test in test_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements_test)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()


if csv_path is not None:
    csv_file.close()
    txt_file.close()


starting epoch 0
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5196667909622192
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5195386409759521
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5190877914428711
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5190682411193848
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5197278261184692
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.519722044467926
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5202361345291138
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5205261707305908
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5206750631332397
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5203545093536377
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.

KeyboardInterrupt: 